In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# DGCNN + PRIMEVUL (FIXED FOR CLASS IMBALANCE)
# Dataset: primevul-dgcnn
# ============================================================

import os, json, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, platform, psutil
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------
# DATASET PATH
# --------------------
DATASET_PATH = "/kaggle/input/primevul-dgcnn"

# --------------------
# LOAD PRIMEVUL
# --------------------
def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            data.append({"code": obj["func"], "label": int(obj["target"])})
    return pd.DataFrame(data)

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("\nLabel distribution (train):")
print(train_df["label"].value_counts())

# ============================================================
# CLASS WEIGHTS (🔥 FIX #1)
# ============================================================

counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [counts[1] / counts.sum(), counts[0] / counts.sum()],
    dtype=torch.float
).to(device)

print("\nClass weights:", class_weights)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
MAX_LEN = 256

class PrimeVulDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.codes[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(PrimeVulDataset(train_df), batch_size=32, shuffle=True)
test_loader  = DataLoader(PrimeVulDataset(test_df), batch_size=32)

# ============================================================
# DGCNN MODEL
# ============================================================

class DGCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, k=8):
        super().__init__()
        self.k = k
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 128, 3, padding=1)
        self.conv3 = nn.Conv1d(128, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * k, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 2)

    def kmax(self, x):
        x, _ = torch.topk(x, self.k, dim=2)
        return x

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.kmax(x).view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = DGCNN(tokenizer.vocab_size).to(device)

# ============================================================
# TRAINING SETUP (🔥 FIX #2)
# ============================================================

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
EPOCHS = 8

# ============================================================
# TRAIN
# ============================================================

print("\nTraining DGCNN...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        loss = criterion(model(ids), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ============================================================
# EVALUATION (🔥 FIX #3: VERIFY BOTH CLASSES)
# ============================================================

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        preds = torch.argmax(model(ids), dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nPrediction distribution:", np.unique(y_pred, return_counts=True))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n===== FIXED DGCNN PRIMEVUL RESULTS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))
print("FPR      :", fp / (fp + tn) if (fp + tn) > 0 else 0)
print("Confusion Matrix:", tn, fp, fn, tp)


Device: cuda

Label distribution (train):
label
1    3789
0    3789
Name: count, dtype: int64

Class weights: tensor([0.5000, 0.5000], device='cuda:0')

Training DGCNN...
Epoch 1/8 | Loss: 0.6947
Epoch 2/8 | Loss: 0.6936
Epoch 3/8 | Loss: 0.6935
Epoch 4/8 | Loss: 0.6937
Epoch 5/8 | Loss: 0.6933
Epoch 6/8 | Loss: 0.6932
Epoch 7/8 | Loss: 0.6933
Epoch 8/8 | Loss: 0.6933

Prediction distribution: (array([0, 1]), array([775,  95]))

===== FIXED DGCNN PRIMEVUL RESULTS =====
Accuracy : 0.5080459770114942
Precision: 0.5368421052631579
Recall   : 0.11724137931034483
F1 Score : 0.19245283018867926
FPR      : 0.10114942528735632
Confusion Matrix: 391 44 384 51
